# IKG SQL Lineage Extraction

Use this notebook to orchestrate the `ikg_sql_lineage.py` workflow from Python. It keeps every step (GitLab sync, SQL parsing, metadata lookups, and exports) reproducible from a single place.

## Prerequisites
- Run `pip install -r requirements.txt` (or manually install `sqlglot`, `python-gitlab`, `pandas`, `openpyxl`, `networkx`, `psycopg2-binary`, `typer`, `xlwt`).
- Ensure you have a valid GitLab private token with read access to the `ikg-dags` project.
- Greenplum credentials (including password) are mandatory; metadata lookups always execute and need `SELECT` on `information_schema.columns` for the configured schemas.

In [ ]:
from pathlib import Path
import os

CONFIG = {
    "GITLAB_URL": os.environ.get("GITLAB_URL", "https://devcloud.ubs.net"),
    "PROJECT_ID": os.environ.get("PROJECT_ID", "ubs/gwma/smart-technology-and-analytics/staat-data-science/staat-ds-genesis/genesis-platform/ikg-dags"),
    "BRANCH": os.environ.get("BRANCH", "ikg-master"),
    "SQL_FOLDER": os.environ.get("SQL_FOLDER", "dags/ikg/scripts/sql"),
    "EXCLUDE_FOLDER": os.environ.get("EXCLUDE_FOLDER", "ikg_new fa_shhp_map"),
    "TOKEN": os.environ.get("TOKEN", ""),
    "START_TABLE": os.environ.get("START_TABLE", "account_profile_curr_ikg"),
    "OUTPUT_DIR": Path(os.environ.get("OUTPUT_DIR", Path.cwd())),
    "GREENPLUM_HOST": os.environ.get("GREENPLUM_HOST", "greenplum-rdsp.zur.swissbank.com"),
    "GREENPLUM_PORT": int(os.environ.get("GREENPLUM_PORT", 5432)),
    "GREENPLUM_DB": os.environ.get("GREENPLUM_DB", "gprdsp"),
    "GREENPLUM_USER": os.environ.get("GREENPLUM_USER", "ds_rdsp_dev"),
    "GREENPLUM_PASSWORD": os.environ.get("GREENPLUM_PASSWORD", ""),
    "GREENPLUM_SCHEMA": os.environ.get("GREENPLUM_SCHEMA", "core_wma_shared,core_model,core_ikg,core_etl"),
}
CONFIG

In [ ]:
from getpass import getpass

while not CONFIG["TOKEN"]:
    CONFIG["TOKEN"] = getpass("GitLab private token (required): ").strip()

while not CONFIG["GREENPLUM_PASSWORD"]:
    CONFIG["GREENPLUM_PASSWORD"] = getpass(
        "Greenplum password (required for metadata disambiguation): "
    ).strip()

CONFIG

In [ ]:
from ikg_sql_lineage import (
    GitLabSQLFetcher,
    GreenplumMetadata,
    SQLLineageParser,
    LineageExporter,
    _split_exclude_folders,
    _split_schemas,
)

CACHE_DIR = Path.cwd() / ".ikg_sql_cache_notebook"
CACHE_DIR.mkdir(exist_ok=True)

def build_metadata(config: dict) -> GreenplumMetadata:
    return GreenplumMetadata(
        host=config["GREENPLUM_HOST"],
        port=config["GREENPLUM_PORT"],
        database=config["GREENPLUM_DB"],
        user=config["GREENPLUM_USER"],
        password=config["GREENPLUM_PASSWORD"],
        schemas=_split_schemas(config["GREENPLUM_SCHEMA"]),
    )


def run_lineage(config: dict) -> dict:
    metadata = build_metadata(config)
    metadata.ensure_connection()
    fetcher = GitLabSQLFetcher(
        gitlab_url=config["GITLAB_URL"],
        private_token=config["TOKEN"],
        project_id=config["PROJECT_ID"],
        branch=config["BRANCH"],
        sql_folder=config["SQL_FOLDER"],
        exclude_folders=_split_exclude_folders(config["EXCLUDE_FOLDER"]),
    )
    repository = fetcher.download(CACHE_DIR)
    parser = SQLLineageParser(
        repository=repository,
        metadata=metadata,
        start_table=config["START_TABLE"].lower(),
        preferred_start_subfolder="ikg_create_profiles",
    )
    records = parser.extract()
    exporter = LineageExporter(records, Path(config["OUTPUT_DIR"]), config["START_TABLE"])
    outputs = exporter.export()
    metadata.close()
    return outputs

In [ ]:
%%time
print("⚠️ Connecting to GitLab and Greenplum …")
outputs = run_lineage(CONFIG)
outputs

In [ ]:
import pandas as pd

if "outputs" in locals() and "xls" in outputs:
    display(pd.read_excel(outputs["xls"]).head())
else:
    print("Run the lineage extraction cell first to materialize outputs.")

After the exports are produced you will find three artifacts per run:

1. `*.xls` – tabular column lineage with target/source schema-table-column and derivation logic.
2. `*.json` – machine-readable copy of the same lineage rows.
3. `*.gexf` – NetworkX graph for downstream visualization (Gephi, Cytoscape, etc.).